<a href="https://colab.research.google.com/github/TottiPuc/LLMs/blob/main/Prompt_Engineering_with_Meta_Llama_3_2_1B_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Llama 3.2 1B SLM with Python & HuggingFace for diverse real-world tasks using Prompt Engineering

In this notebook you will use Meta Llama 3.2 1B SLM via HuggingFace to solve:

- Task 1: Zero-shot Classification
- Task 2: Few-shot Classification
- Task 3: Coding Tasks - Python
- Task 4: Coding Tasks - SQL
- Task 5: Information Extraction
- Task 6: Closed-Domain Question Answering
- Task 7: Open-Domain Question Answering
- Task 8: Document Summarization
- Task 9: Transformation
- Task 10: Translation



___Created By: Dipanjan (DJ)___

## Install HuggingFace dependencies

In [1]:
!pip install transformers==4.46.2
!pip install huggingface_hub==0.26.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.31.0
    Uninstalling huggingface_hub-1.31.0:
      Successfully uninstalled huggingface_hub-1.31.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

## Load Hugging Face Access Token


In [2]:
from getpass import getpass

hf_key = getpass("Enter HuggingFace API Key: ")

Enter HuggingFace API Key: ··········


## Configure Key in Environment


In [3]:
import os

os.environ["HF_TOKEN"] = hf_key

## Make sure you have access to Llama 3.2 1B on HuggingFace

Certain LLMs are gated like [Meta Llama 3.2 1B Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) so make sure to apply for access as shown below else you will get an error when using the model.

__Note: Sometimes it might take a day to get approval from Meta and if you old token still doesn't work, generate a new access token as you did before and use it.__

![](https://i.imgur.com/M88MOu5.png)

## Using LLMs via Hugging Face Inference Client

Thankfully HuggingFace has made its new [__Inference Client__](https://huggingface.co/docs/huggingface_hub/en/package_reference/inference_client) free to use with some basic rate limits etc. in place so you don't end up making unlimited requests on its servers.

The best part is you can access 150,000+ deep learning models without worrying about your infrastructure. Similar to the inference API

In [11]:
import requests

def get_completion(
    prompt,
    api_key=hf_key,
    model="meta-llama/Llama-3.1-8B-Instruct:novita"
):
    response = requests.post(
        "https://router.huggingface.co/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 512,
        },
        timeout=120,
    )

    if not response.ok:
        raise RuntimeError(
            f"Error HTTP {response.status_code}\n"
            f"Detalle de Hugging Face: {response.text}"
        )

    return response.json()["choices"][0]["message"]["content"]

## Let's try out the Llama 3.2 1B model!

In [12]:
from IPython.display import Markdown, display

response = get_completion(prompt='Explain Generative AI in 2 bullet points')
display(Markdown(response))

Here are two bullet points explaining Generative AI:

• **Definition:** Generative AI refers to a type of artificial intelligence that can create new, original content such as images, music, videos, text, or even entire worlds. This is achieved through complex algorithms and machine learning models that can generate new data based on patterns and structures learned from existing data.

• **Applications:** Generative AI has a wide range of applications, including but not limited to: generating realistic synthetic data for training machine learning models, creating artistic content like music or paintings, developing virtual characters or environments for gaming or film, and even generating personalized product recommendations or chatbot responses.

## Task 1: Zero-shot Classification

This prompt tests an LLM's text classification capabilities by prompting it to classify a piece of text without providing any examples.



In [13]:
reviews = [
    f"""
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    """,
    f"""
    Needed a new kitchen blender, but this model has been a nightmare.
    It's supposed to handle various foods, but it struggles with anything tougher than cooked vegetables.
    It's also incredibly noisy, and the 'easy-clean' feature is a joke; food gets stuck under the blades constantly.
    I thought the brand meant quality, but this product has proven me wrong.
    Plus, it arrived three days late. Definitely not worth the expense.
    """
]

In [14]:
responses = []

for review in reviews:
  prompt = f"""
              Act as a product review analyst.
              Given the following review,
              Display the overall sentiment for the review as only one of the following:
              Positive, Negative OR Neutral

              ```{review}```
              """
  response = get_completion(prompt)
  responses.append(response)

In [15]:
for review, response in zip(reviews, responses):
  print('Review:', review)
  print('Sentiment:', response)
  print('------')
  print('\n')

Review: 
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    
Sentiment: Based on the review, I would classify the overall sentiment as **Positive**.

The reviewer uses positive adjectives such as "fantastic", "impressively clear", "right amount of bass", "compact", "bargain", and "on point" to describe the product. They also mention specific strengths of the product, such as its waterproof design and ability to fill a space with sound. The only neutral statement is about the shipping, which is presented as a positive aspect.
------


Review: 
    Needed a new kitchen blender, but this model has

## Task 2: Few-shot Classification

This prompt tests an LLM's text classification capabilities by prompting it to classify a piece of text by providing a few examples of inputs and outputs.



In [16]:
responses = []

for review in reviews:
  prompt = f"""
              Act as a product review analyst.
              Given the following review,
              Display only the overall sentiment for the review:

              Try to classify it by using the following examples as a reference:

              Review: Just received the Laptop I ordered for work, and it's amazing.
              Sentiment: 😊

              Review: Needed a new mechanical keyboard, but this model has been totally disappointing.
              Sentiment: 😡

              Review: ```{review}```
              """
  response = get_completion(prompt)
  responses.append(response)

In [17]:
for review, response in zip(reviews, responses):
  print('Review:', review)
  print('Sentiment:', response)
  print('------')
  print('\n')

Review: 
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    
Sentiment: Overall Sentiment: 😊
------


Review: 
    Needed a new kitchen blender, but this model has been a nightmare.
    It's supposed to handle various foods, but it struggles with anything tougher than cooked vegetables.
    It's also incredibly noisy, and the 'easy-clean' feature is a joke; food gets stuck under the blades constantly.
    I thought the brand meant quality, but this product has proven me wrong.
    Plus, it arrived three days late. Definitely not worth the expense.
    
Sentiment: Sentiment: 😡
------




## Task 3: Coding Tasks - Python

This prompt tests an LLM's capabilities for generating python code based on various tasks


In [18]:
prompt = f"""
Act as an expert in generating python code

Your task is to generate python code
to build a Chain of Though prompt pattern using
the openai python library
"""
response = get_completion(prompt)

In [19]:
from IPython.display import Markdown, display

display(Markdown(response))

Here's an example of how to implement the Chain of Thought (CoT) prompt pattern using the OpenAI Python library.

**Chain of Thought Prompt Pattern**

The Chain of Thought (CoT) prompt pattern is a technique for generating explanations or reasoning behind a model's predictions. This pattern is particularly useful for tasks that require multiple steps, such as mathematical derivations or logical deductions.

**Code Implementation**

```python
from openai import api_key
from openai.embeddings_utils import get_embedding
from openai.engine import create_engine

# Set up OpenAI API key
openai.api_key = api_key

# Create an OpenAI Engine
engine = create_engine("text-davinci-003")

def chain_of_thought_prompt(prompt, num_steps, temperature=0.5):
    """
    Generate a Chain of Thought (CoT) prompt.

    Args:
    - prompt (str): The input prompt.
    - num_steps (int): The number of steps in the CoT prompt.
    - temperature (float): The temperature for the CoT prompt generation. Defaults to 0.5.

    Returns:
    - str: The generated CoT prompt.
    """
    cot_prompt = f"Use the following steps to solve the problem: \n"
    
    for i in range(num_steps):
        # Generate a step in the CoT prompt
        step_prompt = f"Step {i+1}:\n"
        
        # Get the embedding of the previous step
        if i == 0:
            step_prompt += f"Given: {prompt}\n\n"
        else:
            step_prompt += f"Given the previous step, {cot_prompt.strip()}\n\n"

        # Generate the next step in the CoT prompt
        response = engine.complete(
            prompt=step_prompt,
            max_tokens=2048,
            temperature=temperature,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        cot_prompt += response.choices[0].text + "\n\n"

    return cot_prompt

# Example usage
prompt = "What is the value of x in the equation 2x + 5 = 11?"
num_steps = 3
cot_prompt = chain_of_thought_prompt(prompt, num_steps)
print(cot_prompt)
```

**Explanation**

In this code, we define a function `chain_of_thought_prompt` that takes a prompt, the number of steps in the CoT prompt, and

## Task 4: Coding Tasks - SQL

This prompt tests an LLM's capabilities for generating SQL code based on various tasks

In [20]:
prompt = f"""
Act as an expert in generating SQL code.

Understand the following schema of the database tables carefully:
Table departments, columns = [DepartmentId, DepartmentName]
Table employees, columns = [EmployeeId, EmployeeName, DepartmentId]
Table salaries, columns = [EmployeeId, Salary]

Create a MySQL query for the employee with max salary in the 'IT' Department.
"""
response = get_completion(prompt)

In [21]:
display(Markdown(response))

**SQL Query: Get Employee with Max Salary in 'IT' Department**

```sql
SELECT e.EmployeeName, s.Salary
FROM employees e
JOIN salaries s ON e.EmployeeId = s.EmployeeId
JOIN departments d ON e.DepartmentId = d.DepartmentId
WHERE d.DepartmentName = 'IT'
ORDER BY s.Salary DESC
LIMIT 1;
```

**How the Query Works:**

1. Join the `employees`, `salaries`, and `departments` tables on their respective IDs.
2. Filter the results to only include employees in the 'IT' department.
3. Order the results by the salary in descending order to get the employee with the highest salary.
4. Use the `LIMIT 1` clause to return only the top row, which corresponds to the employee with the maximum salary.

**Assumptions:**

* The `DepartmentId` in the `employees` table references the `DepartmentId` in the `departments` table.
* The `EmployeeId` in the `salaries` table references the `EmployeeId` in the `employees` table.
* There is only one employee with the maximum salary in the 'IT' department. If there are multiple employees with the same maximum salary, this query will return one of them.

**Note:** If you want to handle the case where there are multiple employees with the same maximum salary, you can use the `RANK()` or `DENSE_RANK()` function to assign a rank to each row based on the salary, and then select the row with rank 1.

## Task 5: Information Extraction

This prompt tests an LLM's capabilities for extracting and analyzing key entities from documents

In [22]:
clinical_note = """
60-year-old man in NAD with a h/o CAD, DM2, asthma, pharyngitis, SBP,
and HTN on altace for 8 years awoke from sleep around 1:00 am this morning
with a sore throat and swelling of the tongue.
He came immediately to the ED because he was having difficulty swallowing and
some trouble breathing due to obstruction caused by the swelling.
He did not have any associated SOB, chest pain, itching, or nausea.
He has not noticed any rashes.
He says that he feels like it is swollen down in his esophagus as well.
He does not recall vomiting but says he might have retched a bit.
In the ED he was given 25mg benadryl IV, 125 mg solumedrol IV,
and pepcid 20 mg IV.
Family history of CHF and esophageal cancer (father).
"""

In [23]:
prompt = f"""
Act as an expert in analyzing and understanding clinical doctor notes in healthcare.
Extract all symptoms only from the clinical note information below.
Differentiate between symptoms that are present vs. absent.
Give me the probability (high/ medium/ low) of how sure you are about the result.
Add a note on the probabilities and why you think so.

Output as a markdown table with the following columns,
all symptoms should be expanded and no acronyms unless you don't know:

Symptoms | Present/Denies | Probability.

Also expand all acronyms.
Output that also as a separate appendix table in Markdown.

Clinical Note:
```{clinical_note}```
"""
response = get_completion(prompt)

In [24]:
display(Markdown(response))

**Symptoms Extraction Table**

| Symptoms | Present/Denies | Probability |
| --- | --- | --- |
| Sore throat | Present | High |
| Swelling of the tongue | Present | High |
| Difficulty swallowing | Present | High |
| Trouble breathing | Present | High |
| Obstruction | Present | High |
| Feels like swelling in esophagus | Present | Medium |
| Vomiting | Absent/Denies | High |
| Retching | Present | Medium |
| Shortness of breath (SOB) | Absent/Denies | Medium |
| Chest pain | Absent/Denies | Medium |
| Itching | Absent/Denies | Low |
| Nausea | Absent/Denies | Medium |
| Rashes | Absent/Denies | High |
| Swallowing issue in esophagus | Present | High |

**Note:** The probabilities are based on the explicit statements in the clinical note. I considered High if the symptom was explicitly mentioned as present or denied, Medium if the symptom was mentioned in a non-explicit way (e.g., "he says he might have retched a bit"), and Low if the symptom was not mentioned at all.

**Acronyms Expansion Table**

| Acronym | Expansion |
| --- | --- |
| NAD | Normal and Duly |
| DM2 | Diabetes Mellitus Type 2 |
| SBP | Systolic Blood Pressure |
| HTN | Hypertension |
| ED | Emergency Department |
| SOB | Shortness of Breath |
| CHF | Congestive Heart Failure |
| DM | Diabetes Mellitus |

**Note:** I couldn't find any expansion for "SBP" in the clinical note, so I assumed it was an expansion for Systolic Blood Pressure, which is a common medical term. If I missed something, please let me know.

## Task 6: Closed-Domain Question Answering

Question Answering (QA) is a natural language processing task which involves generating the desired answer for the given question. Question Answering can be open-domain QA or closed-domain QA depending on whether the LLM is provided with the relevant context or not.

In the case of closed-domain QA, a question along with relevant context is given. Here the context is nothing but the relevant text which ideally should have the answer. Just like a RAG workflow.

In [25]:
report = """
Three quarters (77%) of the population saw an increase in their regular outgoings over the past year,
according to findings from our recent consumer survey. In contrast, just over half (54%) of respondents
had an increase in their salary, which suggests that the burden of costs outweighing income remains for
most. In total, across the 2,500 people surveyed, the increase in outgoings was 18%, three times higher
than the 6% increase in income.

Despite this, the findings of our survey suggest we have reached a plateau. Looking at savings,
for example, the share of people who expect to make regular savings this year is just over 70%,
broadly similar to last year. Over half of those saving plan to use some of the funds for residential
property. A third are saving for a deposit, and a further 20% for an investment property or second home.

But for some, their plans are being pushed back. 9% of respondents stated they had planned to purchase
a new home this year but have now changed their mind. While for many the deposit may be an issue,
the other driving factor remains the cost of the mortgage, which has been steadily rising the last
few years. For those that currently own a property, the survey showed that in the last year,
the average mortgage payment has increased from £668.51 to £748.94, or 12%."""


In [26]:
question = """
How much has the average mortage payment increased in the last year?
"""

prompt = f"""
Using the following context information below please answer the following question
to the best of your ability
Context:
{report}
Question:
{question}
Answer:
"""

In [27]:
response = get_completion(prompt)
display(Markdown(response))

The average mortgage payment has increased from £668.51 to £748.94 in the last year, which is an increase of £80.43. This represents a 12% increase.

In [28]:
question = """
What percentage of people had an increase in salary last year? Show the answer just as a number.
"""

prompt = f"""
Using the following context information below please answer the following question
to the best of your ability
Context:
{report}
Question:
{question}
Answer:
"""
response = get_completion(prompt)
display(Markdown(response))

54%

## Task 7: Open-Domain Question Answering

Question Answering (QA) is a natural language processing task which involves generating the desired answer for the given question.

In the case of open-domain QA, only the question is asked without providing any context or information. Here, the LLM answers the question using the knowledge gained from large volumes of text data during its training. This is basically Zero-Shot QA

In [29]:
prompt = f"""
Please answer the following question to the best of your ability
Question:
What is LangChain?

Answer:
"""

In [30]:
response = get_completion(prompt)
display(Markdown(response))

LangChain is an open-source, Python-based framework for building conversational AI models using LLMs (Large Language Models). It is designed to simplify the process of developing and integrating LLMs into various applications, such as chatbots, virtual assistants, and other conversational interfaces.

LangChain provides a set of pre-built components and tools that enable developers to easily integrate LLMs into their projects, including:

1. **LLM wrappers**: LangChain provides wrappers for popular LLMs, such as LLaMA, OPT, and T5, making it easy to integrate these models into applications.
2. **Conversational flow**: LangChain allows developers to define conversational flows using a simple, Python-based syntax, making it easy to create complex dialogue trees and conversation scripts.
3. **Action-oriented interface**: LangChain provides an action-oriented interface that enables developers to easily integrate LLMs with external systems, such as databases, APIs, and web services.
4. **Model chaining**: LangChain allows developers to chain multiple LLMs together to create complex models that can perform multiple tasks, such as natural language understanding, text generation, and decision-making.

Overall, LangChain is a powerful tool for building conversational AI models and integrating LLMs into various applications, making it a popular choice among developers and researchers working in the field of natural language processing and conversational AI.

In [31]:
prompt = f"""
Please answer the following question to the best of your ability
Question:
What is LangGraph?

Answer:
"""

In [32]:
response = get_completion(prompt)
display(Markdown(response))

LangGraph is a language model graph database. It is a knowledge graph repository that stores language models and their associated data, allowing for the efficient retrieval of information from large language models and the ability to manipulate their outputs.

## Task 8: Document Summarization

Document summarization is a natural language processing task which involves creating a concise summary of the given text, while still capturing all the important information.

In [33]:
doc = """
Coronaviruses are a large family of viruses which may cause illness in animals or humans.
In humans, several coronaviruses are known to cause respiratory infections ranging from the
common cold to more severe diseases such as Middle East Respiratory Syndrome (MERS) and Severe Acute Respiratory Syndrome (SARS).
The most recently discovered coronavirus causes coronavirus disease COVID-19.
COVID-19 is the infectious disease caused by the most recently discovered coronavirus.
This new virus and disease were unknown before the outbreak began in Wuhan, China, in December 2019.
COVID-19 is now a pandemic affecting many countries globally.
The most common symptoms of COVID-19 are fever, dry cough, and tiredness.
Other symptoms that are less common and may affect some patients include aches
and pains, nasal congestion, headache, conjunctivitis, sore throat, diarrhea,
loss of taste or smell or a rash on skin or discoloration of fingers or toes.
These symptoms are usually mild and begin gradually.
Some people become infected but only have very mild symptoms.
Most people (about 80%) recover from the disease without needing hospital treatment.
Around 1 out of every 5 people who gets COVID-19 becomes seriously ill and develops difficulty breathing.
Older people, and those with underlying medical problems like high blood pressure, heart and lung problems,
diabetes, or cancer, are at higher risk of developing serious illness.
However, anyone can catch COVID-19 and become seriously ill.
People of all ages who experience fever and/or  cough associated with difficulty breathing/shortness of breath,
chest pain/pressure, or loss of speech or movement should seek medical attention immediately.
If possible, it is recommended to call the health care provider or facility first,
so the patient can be directed to the right clinic.
People can catch COVID-19 from others who have the virus.
The disease spreads primarily from person to person through small droplets from the nose or mouth,
which are expelled when a person with COVID-19 coughs, sneezes, or speaks.
These droplets are relatively heavy, do not travel far and quickly sink to the ground.
People can catch COVID-19 if they breathe in these droplets from a person infected with the virus.
This is why it is important to stay at least 1 meter) away from others.
These droplets can land on objects and surfaces around the person such as tables, doorknobs and handrails.
People can become infected by touching these objects or surfaces, then touching their eyes, nose or mouth.
This is why it is important to wash your hands regularly with soap and water or clean with alcohol-based hand rub.
Practicing hand and respiratory hygiene is important at ALL times and is the best way to protect others and yourself.
When possible maintain at least a 1 meter distance between yourself and others.
This is especially important if you are standing by someone who is coughing or sneezing.
Since some infected persons may not yet be exhibiting symptoms or their symptoms may be mild,
maintaining a physical distance with everyone is a good idea if you are in an area where COVID-19 is circulating."""

In [34]:
prompt = f"""
You are an expert in generating accurate document summaries.
Generate a summary of the given document.

Document:
{doc}


Constraints: Please start the summary with the delimiter 'Summary'
and limit the summary to 5 lines

Summary:
"""

response = get_completion(prompt)
display(Markdown(response))

Summary: 
Coronaviruses cause illness in animals or humans, with COVID-19 being a recent pandemic. 
The most common symptoms of COVID-19 are fever, dry cough, and tiredness, with some people experiencing mild symptoms. 
Older people and those with underlying medical problems are at higher risk of developing serious illness. 
COVID-19 spreads primarily through person-to-person contact via small droplets and contaminated surfaces. 
Practicing hand and respiratory hygiene, maintaining distance, and regular hand washing can help protect against the virus.

## Task 9: Transformation

You can use LLMs to take an existing document and transform it into other formats of content and even generate training data for fine-tuning or training models

In [35]:
fact_sheet_mobile = """
PRODUCT NAME
Samsung Galaxy Z Fold4 5G Black
​
PRODUCT OVERVIEW
Stands out. Stands up. Unfolds.
The Galaxy Z Fold4 does a lot in one hand with its 15.73 cm(6.2-inch) Cover Screen.
Unfolded, the 19.21 cm(7.6-inch) Main Screen lets you really get into the zone.
Pushed-back bezels and the Under Display Camera means there's more screen
and no black dot getting between you and the breathtaking Infinity Flex Display.
Do more than more with Multi View. Whether toggling between texts or catching up
on emails, take full advantage of the expansive Main Screen with Multi View.
PC-like power thanks to Qualcomm Snapdragon 8+ Gen 1 processor in your pocket,
transforms apps optimized with One UI to give you menus and more in a glance
New Taskbar for PC-like multitasking. Wipe out tasks in fewer taps. Add
apps to the Taskbar for quick navigation and bouncing between windows when
you're in the groove.4 And with App Pair, one tap launches up to three apps,
all sharing one super-productive screen
Our toughest Samsung Galaxy foldables ever. From the inside out,
Galaxy Z Fold4 is made with materials that are not only stunning,
but stand up to life's bumps and fumbles. The front and rear panels,
made with exclusive Corning Gorilla Glass Victus+, are ready to resist
sneaky scrapes and scratches. With our toughest aluminum frame made with
Armor Aluminum, this is one durable smartphone.
World’s first water resistant foldable smartphones. Be adventurous, rain
or shine. You don't have to sweat the forecast when you've got one of the
world's first water-resistant foldable smartphones.
​
PRODUCT SPECS
OS - Android 12.0
RAM - 12 GB
Product Dimensions - 15.5 x 13 x 0.6 cm; 263 Grams
Batteries - 2 Lithium Ion batteries required. (included)
Item model number - SM-F936BZKDINU_5
Wireless communication technologies - Cellular
Connectivity technologies - Bluetooth, Wi-Fi, USB, NFC
GPS - True
Special features - Fast Charging Support, Dual SIM, Wireless Charging, Built-In GPS, Water Resistant
Other display features - Wireless
Device interface - primary - Touchscreen
Resolution - 2176x1812
Other camera features - Rear, Front
Form factor - Foldable Screen
Colour - Phantom Black
Battery Power Rating - 4400
Whats in the box - SIM Tray Ejector, USB Cable
Manufacturer - Samsung India pvt Ltd
Country of Origin - China
Item Weight - 263 g
"""

In [36]:
prompt =f"""Turn the following product description into a list of frequently asked questions (FAQ).
Show both the question and it's corresponding answer.
Create at the max 8 FAQs

Product description:
```{fact_sheet_mobile}```
"""

response = get_completion(prompt)
display(Markdown(response))

Here are 8 FAQs based on the product description:

**Q1: What is the size of the Galaxy Z Fold4's Cover Screen?**
A1: The Galaxy Z Fold4's Cover Screen measures 15.73 cm (6.2 inches).

**Q2: What is the size of the Galaxy Z Fold4's Main Screen when unfolded?**
A2: The Galaxy Z Fold4's Main Screen measures 19.21 cm (7.6 inches) when unfolded.

**Q3: What type of display technology does the Galaxy Z Fold4 have?**
A3: The Galaxy Z Fold4 features an Infinity Flex Display with an Under Display Camera, allowing for a more seamless viewing experience.

**Q4: Can I multitask on the Galaxy Z Fold4?**
A4: Yes, the Galaxy Z Fold4 allows for PC-like multitasking with features like Multi View, Taskbar, and App Pair, making it easy to switch between apps and tasks.

**Q5: Is the Galaxy Z Fold4 water-resistant?**
A5: Yes, the Galaxy Z Fold4 is one of the world's first water-resistant foldable smartphones, making it perfect for adventurous users.

**Q6: What is the processor speed of the Galaxy Z Fold4?**
A6: The Galaxy Z Fold4 is powered by a Qualcomm Snapdragon 8+ Gen 1 processor, providing fast and efficient performance.

**Q7: What type of battery does the Galaxy Z Fold4 have?**
A7: The Galaxy Z Fold4 comes with 2 Lithium Ion batteries, providing a battery power rating of 4400.

**Q8: What colors is the Galaxy Z Fold4 available in?**
A8: The Galaxy Z Fold4 is available in Phantom Black.

## Task 10: Translation

You can use LLMs to take an existing document and translate it from a source to target language. You can also translate to multiple languages at the same time

In [37]:
prompt = """You are an expert translator.
Translate the given text from English to German and Spanish.

Text: 'Hello, how are you today?'
Translation:
"""

response = get_completion(prompt)
display(Markdown(response))

Here are the translations of the given text into German and Spanish:

1. German Translation:
 Hallo, wie geht es Ihnen heute?

Breakdown:
- 'Hello' translates to 'Hallo'
- 'how' translates to 'wie'
- 'are you' translates to 'geht es'
- 'you' translates to 'Ihnen' (formal) or 'dir' (informal)
- 'today' translates to 'heute'

2. Spanish Translation:
 Hola, ¿cómo estás hoy?

Breakdown:
- 'Hello' translates to 'Hola'
- 'how' translates to '¿cómo'
- 'are you' translates to 'estás'
- 'you' translates to 'tú' or 'usted' (formal)
- 'today' translates to 'hoy'